# DATA 612 — Benchmark data prep: YOOCHOOSE (RecSys Challenge 2015), session-level

Converts YOOCHOOSE click/buy sessions into **the same tables and columns as the Criteo pipeline**, using the
repo's own windowing and label code (`build_windowed_dataset`), so the training notebooks can read it.

**YOOCHOOSE is session-based: there are no user IDs.** That needs a different time setup from the other datasets:

| Choice | Why |
|---|---|
| sequence = **session** (`uid` = session ID) | no persistent users |
| **5-minute windows, timed from each session's start** | sessions last minutes; a global calendar would put a whole session in one window |
| sessions capped at **2 hours** (later events dropped, counted) | a few outlier sessions run for days |
| **30-minute** inactivity rule (instead of 7 days) | a session does not come back after it ends |
| **chronological split by session start date** (70/15/15 of the 6 months) | window numbers restart in every session, so they cannot define a time split |
| `user_group` = hashed session ID | gives a "session-disjoint" split, the counterpart of user-disjoint |

**Mapping:** impression = click (and purchase), campaign token = item category (top `MAX_CATEGORIES` + `other`
+ `unknown`; "S" = special offer is its own token), click = purchase event, conversion = **buy**, cost = 0.

**Runtime:** CPU **with High-RAM**. ~33M clicks.

## 1. Settings

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")
    REPO_DIR  = Path("/content/repo")
    WORK_DIR  = Path("/content/work")
else:
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()
    WORK_DIR  = Path(os.environ.get("WORK_DIR", "work")).resolve()

WINDOW_MINUTES     = 5
INACTIVE_MINUTES   = 30       # a session is a prediction position up to 30 min after its last activity
MAX_SESSION_HOURS  = 2
MAX_CATEGORIES     = 675      # same vocabulary size as Criteo; rarer categories -> "other"
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
N_SHARDS = 16
RAW_DIR = DRIVE_DIR / "raw" / "yoochoose"
NAME    = f"yoochoose_w{WINDOW_MINUTES}m"
RAW_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Raw files:", RAW_DIR, "| output name:", NAME)

## 2. Repo (the shared pipeline) and packages

In [ ]:
REPO_URL = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))

need = [p for p, m in {"pyarrow": "pyarrow", "py7zr": "py7zr"}.items() if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import numpy as np, pandas as pd, yaml
from src.data.preprocess import DataColumns, build_campaign_vocab, build_windowed_dataset, finalize_meta, \
    n_windows_for, write_table
from src.data.splits import window_boundaries
import prepare_criteo as pc          # reuse: add_split_columns, split_counts, merge_counts, shard_ids, TABLES
cfg = yaml.safe_load((REPO_DIR / "configs" / "base.yaml").read_text())
SEED = cfg["project"].get("seed", 42)
print("pandas", pd.__version__, "| seed", SEED)

## 3. Raw files (once, to Drive)
Needed: `yoochoose-clicks.dat` and `yoochoose-buys.dat` (`yoochoose-test.dat` is not used: it has no buys).
This cell tries the original archive (`yoochoose-data.7z`, the link used by the beta-recsys library). If that
link no longer works, download the RecSys Challenge 2015 data yourself (e.g. a Kaggle mirror of
"RecSys Challenge 2015 / YOOCHOOSE"), put the two `.dat` files (or the `.7z`) in
`MyDrive/DATA612_data/raw/yoochoose/`, and re-run.

In [ ]:
NEED = ["yoochoose-clicks.dat", "yoochoose-buys.dat"]
URL = "https://s3-eu-west-1.amazonaws.com/yc-rdata/yoochoose-data.7z"
archive = RAW_DIR / "yoochoose-data.7z"
if not all((RAW_DIR / f).exists() for f in NEED):
    if not archive.exists():
        try:
            print("Downloading", URL)
            urllib.request.urlretrieve(URL, archive)
        except Exception as e:
            print(f"Automatic download failed ({e}). Put the files in {RAW_DIR} manually (see above).")
    if archive.exists():
        import py7zr
        with py7zr.SevenZipFile(archive) as z:
            names = [n for n in z.getnames() if Path(n).name in NEED]
            z.extract(path=WORK_DIR / "yc_extract", targets=names)
        for n in names:
            shutil.copy2(WORK_DIR / "yc_extract" / n, RAW_DIR / Path(n).name)
missing = [f for f in NEED if not (RAW_DIR / f).exists()]
assert not missing, f"Missing {missing} in {RAW_DIR}"
for f in NEED:                                  # local copies: much faster than reading from Drive
    if not (WORK_DIR / f).exists():
        shutil.copy2(RAW_DIR / f, WORK_DIR / f)
    print(f"{f:<24} {(WORK_DIR / f).stat().st_size / 1e9:.2f} GB")

## 4. Map sessions to the Criteo input schema (session-relative time)

In [ ]:
import pyarrow as pa, pyarrow.csv as pacsv
t0 = time.time()
ts_type = pa.timestamp("ms", tz="UTC")
clicks = pacsv.read_csv(WORK_DIR / NEED[0],
    read_options=pacsv.ReadOptions(column_names=["session", "ts", "item", "category"]),
    convert_options=pacsv.ConvertOptions(column_types={"session": pa.int64(), "ts": ts_type,
                                                       "item": pa.int64(), "category": pa.string()})).to_pandas()
buys = pacsv.read_csv(WORK_DIR / NEED[1],
    read_options=pacsv.ReadOptions(column_names=["session", "ts", "item", "price", "quantity"]),
    convert_options=pacsv.ConvertOptions(column_types={"session": pa.int64(), "ts": ts_type, "item": pa.int64(),
                                                       "price": pa.float64(), "quantity": pa.int64()})).to_pandas()
print(f"clicks: {len(clicks):,} | buys: {len(buys):,} | sessions: {clicks['session'].nunique():,} | "
      f"buying sessions: {buys['session'].nunique():,} | read in {time.time() - t0:.0f}s")
print("period:", clicks["ts"].min(), "->", clicks["ts"].max())

# Category token: "0" = unknown (-1), "S" = special offer (-3), numbers = category / brand id; rare ones -> other (-2).
cat = clicks["category"].astype("category")
cat_vals = cat.cat.categories.astype(str)
cat_num = pd.to_numeric(pd.Series(cat_vals), errors="coerce").to_numpy()
code_val = np.where(cat_vals == "S", -3, np.where(np.isnan(cat_num), -1, np.nan_to_num(cat_num, nan=-1))).astype(np.int64)
code_val[code_val == 0] = -1
tok = code_val[cat.cat.codes.to_numpy()]
counts = pd.Series(tok[tok >= 0]).value_counts()
keep = counts.index[:MAX_CATEGORIES - 3].to_numpy()                     # leave room for unknown / other / special
tok = np.where((tok >= 0) & ~np.isin(tok, keep), -2, tok)
print(f"distinct category values: {len(cat_vals):,} -> tokens: {len(np.unique(tok))} "
      f"(unknown {(tok == -1).mean():.1%}, special offer {(tok == -3).mean():.1%}, other {(tok == -2).mean():.1%})")
clicks["token"] = tok
del cat, tok

# A purchased item's token = its most frequent token among clicks (unknown if never clicked).
item_tok = (clicks.groupby(["item", "token"]).size().rename("n").reset_index()
                  .sort_values(["item", "n"], ascending=[True, False]).drop_duplicates("item")
                  .set_index("item")["token"])
buys["token"] = buys["item"].map(item_tok).fillna(-1).astype(np.int64)

ev = pd.concat([
    pd.DataFrame({"session": clicks["session"], "ts": clicks["ts"], "token": clicks["token"], "buy": np.int8(0)}),
    pd.DataFrame({"session": buys["session"], "ts": buys["ts"], "token": buys["token"], "buy": np.int8(1)}),
], ignore_index=True)
del clicks, buys, item_tok
ev["sec"] = ev["ts"].dt.tz_convert(None).astype("datetime64[ms]").astype("int64") // 1000   # unix seconds
start = ev.groupby("session")["sec"].transform("min")
ev["rel"] = (ev["sec"] - start).astype(np.int64)                          # seconds since the session started
cap = MAX_SESSION_HOURS * 3600
N_CAPPED = int((ev["rel"] >= cap).sum())
N_CAPPED_BUYS = int(((ev["rel"] >= cap) & (ev["buy"] == 1)).sum())
ev = ev[ev["rel"] < cap]
session_start = ev.groupby("session")["sec"].min()
print(f"dropped {N_CAPPED:,} events ({N_CAPPED_BUYS:,} buys) after {MAX_SESSION_HOURS}h of a session")

ev = ev.sort_values(["session", "rel"], kind="mergesort").reset_index(drop=True)
is_buy = ev["buy"].to_numpy() == 1
raw = pd.DataFrame({
    "timestamp": ev["rel"].to_numpy(np.int64),
    "uid": ev["session"].to_numpy(np.int64),
    "campaign": ev["token"].to_numpy(np.int64),
    "conversion": is_buy.astype(np.int8),
    "conversion_timestamp": np.where(is_buy, ev["rel"], -1).astype(np.int64),
    "conversion_id": np.where(is_buy, np.arange(len(ev)), -1).astype(np.int64),
    "click": is_buy.astype(np.int8),
    "cost": np.zeros(len(ev), dtype=np.float32),
})
del ev
print(f"mapped to Criteo schema in {time.time() - t0:.0f}s:", raw.shape)
raw.head()

## 5. Build windows and labels (repo pipeline), then the session-level splits
`time_split` comes from each session's **start date** (not from window numbers); `user_group` is the repo's
hashed split applied to session IDs.

In [ ]:
cols = DataColumns.from_config(cfg["data"])
horizon = int(cfg["windowing"].get("horizon_windows", 1))
hours, inactive_hours = WINDOW_MINUTES / 60, INACTIVE_MINUTES / 60

span0, span1 = int(session_start.min()), int(session_start.max())
train_end = span0 + TRAIN_FRAC * (span1 - span0)
val_end = span0 + (TRAIN_FRAC + VAL_FRAC) * (span1 - span0)
sess_split = pd.Series(np.where(session_start < train_end, "train",
                                np.where(session_start < val_end, "val", "test")), index=session_start.index)
to_date = lambda s: str(pd.to_datetime(int(s), unit="s").date())
SPLIT_DATES = {"train": [to_date(span0), to_date(train_end)], "val": [to_date(train_end), to_date(val_end)],
               "test": [to_date(val_end), to_date(span1)]}
print("Session-start split:", SPLIT_DATES)

vocab = build_campaign_vocab(raw, cols)
shards = pc.shard_ids(raw[cols.user], N_SHARDS, SEED)
n_windows = n_windows_for(raw, hours, cols)
bounds = window_boundaries(n_windows, 0.70, 0.15, horizon)              # only used for the user_group column
out_dir = WORK_DIR / "processed" / NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
write_table(vocab, out_dir / "campaign_vocab.parquet")

t0 = time.time()
totals, counts, meta = {k: 0 for k in pc.SUMMED_META}, {}, {}
for i in range(N_SHARDS):
    part = raw[shards == i]
    if part.empty:
        continue
    res = build_windowed_dataset(part, hours, inactive_hours, horizon, cols,
                                 vocab=vocab, n_windows=n_windows, timestamp_unit="seconds")
    p = pc.add_split_columns(res["positions"], bounds, cfg, SEED)          # adds user_group (hashed sessions)
    p["time_split"] = p["uid"].map(sess_split).astype("category")         # chronological = session start date
    res["positions"] = p
    counts = pc.merge_counts(counts, pc.split_counts(p))
    for t in pc.TABLES:
        write_table(res[t], out_dir / t / f"part-{i:03d}.parquet")
    meta = res["meta"]
    for k in pc.SUMMED_META:
        totals[k] += meta[k]
    print(f"  shard {i + 1}/{N_SHARDS}: {meta['n_positions']:,} positions")

meta.update(totals)
meta["n_campaigns"] = int(len(vocab))
meta = finalize_meta(meta)
meta["splits"] = {"boundaries": None, "session_start_dates": SPLIT_DATES, **counts}
meta.update({"seed": SEED, "n_shards": N_SHARDS, "source_dataset": "yoochoose_recsys2015",
             "unit": "session (uid = session id)", "time_axis": "session-relative (window 0 = session start)",
             "window_minutes": WINDOW_MINUTES, "inactive_minutes": INACTIVE_MINUTES,
             "max_session_hours": MAX_SESSION_HOURS, "events_dropped_after_cap": N_CAPPED,
             "buys_dropped_after_cap": N_CAPPED_BUYS,
             "time_split_rule": f"session start date, {TRAIN_FRAC:.0%}/{VAL_FRAC:.0%}/{1 - TRAIN_FRAC - VAL_FRAC:.0%} of the period",
             "campaign_token": f"item category (top {MAX_CATEGORIES - 3} + other + unknown + special offer)",
             "conversion": "buy", "click": "purchase event", "cost": "not available (0)"})
(out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
print(f"built in {time.time() - t0:.0f}s")

## 6. Sanity checks

In [ ]:
pos = pd.read_parquet(out_dir / "positions")
summary = {
    "window": f"{WINDOW_MINUTES} min (session-relative)", "windows per session (max)": meta["n_windows"],
    "events": meta["n_impressions"], "sessions": meta["n_users"], "tokens (categories)": meta["n_campaigns"],
    "distinct buys": meta["n_conversions_total"], "positions": meta["n_positions"],
    "positive positions": meta["n_positive_positions"], "positive rate": f"{meta['positive_rate_conversion']:.4%}",
    "positions with a next-window category": f"{meta['frac_positions_with_next_campaign']:.1%}",
    "buy windows covered": f"{meta['frac_conversion_windows_covered']:.1%}",
}
display(pd.Series(summary, name=NAME).to_frame())
display(pd.DataFrame(meta["splits"]["time_split"]).T.rename_axis("time_split (session start)"))
display(pd.DataFrame(meta["splits"]["user_group"]).T.rename_axis("user_group (hashed sessions)"))

sess_splits = pos.groupby("uid")["time_split"].nunique()
checks = {
    "labels come from the next window": bool((pos["label_window"] == pos["window"] + horizon).all()),
    "each session is in exactly one time split": bool((sess_splits == 1).all()),
    "positives exist in every split": all(meta["splits"]["time_split"][s]["n_positive"] > 0 for s in ("train", "val", "test")),
    "same columns as Criteo positions": set(pos.columns) >= {"uid", "window", "label_window", "is_active",
        "windows_since_active", "y_conversion", "n_next_campaigns", "time_split", "user_group"},
}
for k, v in checks.items():
    print(("OK      " if v else "PROBLEM ") + k)

## 7. Copy to Drive

In [ ]:
dst = DRIVE_DIR / "processed" / NAME
t0 = time.time()
shutil.copytree(out_dir, dst, dirs_exist_ok=True)
size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
print(f"{NAME} -> {dst} ({size:.2f} GB, {time.time() - t0:.0f}s)")
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive synced and unmounted.")

## Notes for the report
- YOOCHOOSE is **session-level** (no user IDs): "user-disjoint" becomes **session-disjoint**, and time is measured
  from each session's start in 5-minute windows. The task is "will this session buy in the next 5 minutes".
- The chronological split is by **session start date**; `meta.json` has the dates. Because window numbers are
  session-relative, the training notebooks must take campaign popularity from `time_split` rather than from window
  numbers, and use session-relative time features. They will be adapted before benchmark training.
- Events after 2 hours of a session are dropped (counts in `meta.json`).